# Daily Challenge – Multi-Head Attention & Natural Language Inference

**Dataset :** Stanford Natural Language Inference (SNLI) via Hugging Face  
**Objectif :** Implémenter une attention simple, puis multi-têtes, et construire un encodeur léger pour la classification NLI (entailment / contradiction / neutral).

## Setup

In [ ]:
%pip install --quiet datasets transformers torch

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', DEVICE)

## Part 1 – Single-Head Attention

### Objectif
Implémenter le building block de base avant de passer au multi-tête.

In [ ]:
class SingleHeadAttention(nn.Module):
    """
    Scaled dot-product attention avec projections linéaires Q/K/V.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.scale = math.sqrt(hidden_dim)

        # Projections linéaires pour Q, K, V
        self.W_q = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_k = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_v = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_o = nn.Linear(hidden_dim, hidden_dim, bias=False)

    def forward(self, x, mask=None):
        # x : (batch, seq_len, hidden_dim)
        Q = self.W_q(x)   # (batch, seq, hidden)
        K = self.W_k(x)
        V = self.W_v(x)

        # Scores d'attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # (batch, seq, seq)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)  # (batch, seq, seq)
        context      = torch.matmul(attn_weights, V)  # (batch, seq, hidden)
        output       = self.W_o(context)

        return output, attn_weights

In [ ]:
# Validation avec des tenseurs factices
batch_size, seq_len, hidden_dim = 2, 6, 32
dummy_input = torch.randn(batch_size, seq_len, hidden_dim)

single_attn = SingleHeadAttention(hidden_dim)
output, attn_weights = single_attn(dummy_input)

print('Input  shape :', dummy_input.shape)
print('Output shape :', output.shape)
print('Attn   shape :', attn_weights.shape)

# Log des poids d'attention (premier exemple du batch)
print('\nAttention weights (sample 0) :')
print(attn_weights[0].detach().numpy().round(3))

## Part 2 – Multi-Head Attention

### Objectif
Étendre l'attention à plusieurs têtes parallèles avec dropout et connexion résiduelle.

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention : découpe hidden_dim en num_heads sous-espaces,
    applique l'attention sur chacun, puis concatène.
    """
    def __init__(self, hidden_dim, num_heads, dropout=0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0, "hidden_dim doit être divisible par num_heads"

        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.scale      = math.sqrt(self.head_dim)

        self.W_q    = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_k    = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_v    = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_o    = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, S, _ = x.shape

        # Projections et reshape : (B, S, H, head_dim) → (B, H, S, head_dim)
        Q = self.W_q(x).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        # Scores : (B, H, S, S)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))

        attn_weights = self.dropout(torch.softmax(scores, dim=-1))
        context      = torch.matmul(attn_weights, V)  # (B, H, S, head_dim)

        # Concaténation des têtes
        context = context.transpose(1, 2).reshape(B, S, self.hidden_dim)
        output  = self.W_o(context)

        # Connexion résiduelle
        output = output + x

        return output, attn_weights

In [ ]:
# Exemple forward avec shapes
hidden_dim = 64
num_heads  = 4
dummy      = torch.randn(2, 10, hidden_dim)

mha = MultiHeadAttention(hidden_dim, num_heads)
out, weights = mha(dummy)

print('Input  shape :', dummy.shape)
print('Output shape :', out.shape)
print('Weights shape:', weights.shape, ' → (batch, num_heads, seq, seq)')

## Part 3 – Encoder + Training Loop sur SNLI

### Objectif
Composer les blocs en un encodeur léger et l'entraîner sur NLI.

In [ ]:
class EncoderBlock(nn.Module):
    """Un bloc Transformer encoder : MHA + LayerNorm + FeedForward + LayerNorm."""
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn     = MultiHeadAttention(hidden_dim, num_heads, dropout)
        self.norm1    = nn.LayerNorm(hidden_dim)
        self.ff       = nn.Sequential(
            nn.Linear(hidden_dim, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, hidden_dim),
        )
        self.norm2    = nn.LayerNorm(hidden_dim)
        self.dropout  = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, _  = self.attn(x, mask)
        x            = self.norm1(x + self.dropout(attn_out))
        ff_out       = self.ff(x)
        x            = self.norm2(x + self.dropout(ff_out))
        return x


class NLIClassifier(nn.Module):
    """Encodeur léger pour NLI (3 classes : entailment, contradiction, neutral)."""
    def __init__(self, vocab_size, hidden_dim=64, num_heads=4, ff_dim=128,
                 num_layers=2, num_classes=3, max_len=64, dropout=0.1):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, hidden_dim, padding_idx=0)
        self.pos_embed  = nn.Embedding(max_len, hidden_dim)
        self.layers     = nn.ModuleList([
            EncoderBlock(hidden_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, attention_mask=None):
        B, S = input_ids.shape
        positions = torch.arange(S, device=input_ids.device).unsqueeze(0)
        x = self.embedding(input_ids) + self.pos_embed(positions)
        for layer in self.layers:
            x = layer(x, attention_mask)
        # Classification sur le premier token (style [CLS])
        pooled = x[:, 0, :]
        return self.classifier(pooled)

In [ ]:
# Chargement du dataset SNLI
print("Chargement SNLI...")
snli = load_dataset('snli', split='train[:5000]')  # sous-ensemble pour la démo
snli_val = load_dataset('snli', split='validation[:1000]')

# Filtrer les exemples sans label (-1 = pas d'accord entre annotateurs)
snli     = snli.filter(lambda x: x['label'] != -1)
snli_val = snli_val.filter(lambda x: x['label'] != -1)

print(f"Train : {len(snli)} | Val : {len(snli_val)}")
print("Exemple :", snli[0])

In [ ]:
# Tokenisation avec BERT tokenizer (vocab partagé)
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN   = 64

def encode_pair(example):
    enc = tokenizer(
        example['premise'],
        example['hypothesis'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
    )
    enc['label'] = example['label']
    return enc

snli_tok     = snli.map(encode_pair)
snli_val_tok = snli_val.map(encode_pair)

snli_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
snli_val_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

train_loader = DataLoader(snli_tok,     batch_size=32, shuffle=True)
val_loader   = DataLoader(snli_val_tok, batch_size=32)

print('Tokenisation terminée.')

In [ ]:
# Entraînement
vocab_size  = tokenizer.vocab_size
nli_model   = NLIClassifier(vocab_size=vocab_size).to(DEVICE)
optimizer   = torch.optim.AdamW(nli_model.parameters(), lr=1e-3)
criterion   = nn.CrossEntropyLoss()

train_losses = []
val_accs     = []

for epoch in range(3):
    nli_model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        mask      = batch['attention_mask'].to(DEVICE)
        labels    = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = nli_model(input_ids, mask)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    # Validation
    nli_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            mask      = batch['attention_mask'].to(DEVICE)
            labels    = batch['label'].to(DEVICE)
            preds     = nli_model(input_ids, mask).argmax(dim=1)
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)

    val_acc = correct / total
    val_accs.append(val_acc)
    print(f'Epoch {epoch+1}/3 | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.3f}')

In [ ]:
# Courbes d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, 4), train_losses, marker='o', color='steelblue')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(range(1, 4), val_accs, marker='s', color='darkorange')
axes[1].set_title('Validation Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')

plt.suptitle('NLI Classifier – Courbes d\'entraînement', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 4 – Visualisation des poids d'attention

In [ ]:
# Visualisation des poids d'attention sur un exemple
sample = snli_val_tok[0]
input_ids = sample['input_ids'].unsqueeze(0).to(DEVICE)
mask      = sample['attention_mask'].unsqueeze(0).to(DEVICE)

nli_model.eval()
# Accès aux poids d'attention du premier EncoderBlock
x = nli_model.embedding(input_ids) + nli_model.pos_embed(
    torch.arange(input_ids.shape[1], device=DEVICE).unsqueeze(0)
)
_, attn_w = nli_model.layers[0].attn(x, mask)
attn_w = attn_w.detach().cpu()

# Tokens décodés
tokens = tokenizer.convert_ids_to_tokens(sample['input_ids'].tolist())
tokens = [t if t != '[PAD]' else '' for t in tokens]
n_tokens = (sample['attention_mask'] == 1).sum().item()
tokens_vis = tokens[:n_tokens]

# Heatmap tête 0
attn_head0 = attn_w[0, 0, :n_tokens, :n_tokens].numpy()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(attn_head0, cmap='Blues', aspect='auto')
ax.set_xticks(range(n_tokens))
ax.set_yticks(range(n_tokens))
ax.set_xticklabels(tokens_vis, rotation=90, fontsize=8)
ax.set_yticklabels(tokens_vis, fontsize=8)
plt.colorbar(im, ax=ax)
ax.set_title('Attention Weights – Tête 0, Couche 0', fontweight='bold')
plt.tight_layout()
plt.show()

## Part 5 – Réflexion

### Comparaison approche custom vs transformer préentraîné

| Aspect | Encodeur custom (ce notebook) | BERT préentraîné fine-tuné |
|---|---|---|
| **Paramètres** | ~1M params (léger) | 110M params (bert-base) |
| **Données nécessaires** | Beaucoup (entraîné from scratch) | Peu (fine-tuning rapide) |
| **Performance NLI** | ~40–50% (limité par la taille et les epochs) | ~90%+ (SOTA) |
| **Temps d'entraînement** | Rapide (CPU quelques minutes) | Lent sans GPU |
| **Interprétabilité attention** | Directe (on inspecte nos propres têtes) | Possible mais plus complexe |

### Observations sur les poids d'attention

- Les premières couches tendent à exhiber une attention **locale** : chaque token regarde surtout ses voisins directs (pattern diagonal ou banded).
- Le token `[CLS]` accumule de l'attention depuis de nombreux tokens, confirmant son rôle d'agrégateur pour la classification.
- Les tokens `[SEP]` et `[PAD]` reçoivent souvent peu d'attention car ils ne portent pas d'information sémantique.
- Avec plus de têtes et de couches, chaque tête se spécialise sur un type de relation différent (syntaxe, coréférence, sémantique).

### Amélioration suggérée
Remplacer l'encodeur custom par un **fine-tuning de `bert-base-uncased`** sur SNLI complet : les embeddings préentraînés permettraient d'atteindre ~90% de validation accuracy en quelques epochs, contre ~45% pour notre modèle entraîné from scratch sur 5000 exemples.